In [4]:
from pathlib import Path
import requests
import geopandas as gpd


# ============================================================
# OUTPUT FOLDER
# ============================================================

script_dir = Path.cwd()

output_dir = script_dir / "Hawaii_StreamStats_All_Gages"
output_dir.mkdir(parents=True, exist_ok=True)

output_shp = output_dir / "Hawaii_StreamStats_All_Gages.shp"
output_geojson = output_dir / "Hawaii_StreamStats_All_Gages.geojson"


# ============================================================
# STREAMSTATS MAP LAYER
# ============================================================

# Actual StreamStats "Streamgages" map layer
url = (
    "https://gis.streamstats.usgs.gov/"
    "arcgis/rest/services/StreamStats/"
    "nationalLayers/MapServer/0/query"
)


# ============================================================
# HAWAII BOUNDING BOX
# ============================================================

# Covers the main Hawaiian Islands:
# Kauai, Oahu, Molokai, Lanai, Maui, Kahoolawe, Hawaii Island

xmin = -160.40
ymin = 18.80
xmax = -154.70
ymax = 22.30


# ============================================================
# QUERY EVERY STREAMSTATS TRIANGLE
# ============================================================

params = {
    "where": "1=1",

    # Hawaii bounding box
    "geometry": f"{xmin},{ymin},{xmax},{ymax}",

    "geometryType": "esriGeometryEnvelope",

    # Bounding box coordinates are lat/lon
    "inSR": "4326",

    # Return output in lat/lon
    "outSR": "4326",

    # Return every attribute
    "outFields": "*",

    # Include point geometry
    "returnGeometry": "true",

    # GeoJSON output
    "f": "geojson"
}


print("Downloading ALL StreamStats triangles in Hawaii...")


response = requests.get(
    url,
    params=params,
    timeout=120
)

response.raise_for_status()


# ============================================================
# CHECK RESPONSE
# ============================================================

data = response.json()

features = data.get("features", [])

print()
print("Number of StreamStats triangles found:")
print(len(features))


if len(features) == 0:
    raise RuntimeError(
        "StreamStats returned zero gages. "
        "Check the API response."
    )


# ============================================================
# SAVE RAW GEOJSON
# ============================================================

with open(
    output_geojson,
    "w",
    encoding="utf-8"
) as f:

    f.write(response.text)


print()
print("Raw GeoJSON saved:")
print(output_geojson)


# ============================================================
# READ INTO GEOPANDAS
# ============================================================

gdf = gpd.read_file(output_geojson)


print()
print("Fields returned by StreamStats:")
print(gdf.columns.tolist())


print()
print("First five records:")
print(gdf.head().to_string())


# ============================================================
# ADD LATITUDE / LONGITUDE
# ============================================================

gdf["longitude"] = gdf.geometry.x
gdf["latitude"] = gdf.geometry.y


# ============================================================
# SAVE SHAPEFILE
# ============================================================

gdf.to_file(
    output_shp,
    driver="ESRI Shapefile"
)


# ============================================================
# SAVE CSV
# ============================================================

gdf.drop(
    columns="geometry"
).to_csv(
    output_dir / "Hawaii_StreamStats_All_Gages.csv",
    index=False
)


# ============================================================
# FINISHED
# ============================================================

print()
print("=" * 60)
print("DONE")
print("=" * 60)

print()
print(
    f"TOTAL HAWAII STREAMSTATS TRIANGLES: {len(gdf)}"
)

print()
print("Shapefile:")
print(output_shp)

print()
print("GeoJSON:")
print(output_geojson)

print()
print("Output folder:")
print(output_dir)


Number of StreamStats triangles found:
493

Raw GeoJSON saved:
C:\Users\bgorberg\Documents\Integral\Get_Stream_Gauges\Hawaii_StreamStats_All_Gages\Hawaii_StreamStats_All_Gages.geojson

Fields returned by StreamStats:
['OBJECTID', 'Latitude', 'Longitude', 'STA_ID', 'STA_NAME', 'STA_TYPE', 'COORDSRC', 'SITESTATUS', 'REACH100', 'MEASURE100', 'REACH24', 'MEASURE24', 'REACHLOC', 'MEASURELOC', 'FeatureURL', 'geometry']

First five records:
   OBJECTID  Latitude  Longitude    STA_ID                                         STA_NAME STA_TYPE COORDSRC SITESTATUS REACH100  MEASURE100 REACH24  MEASURE24 REACHLOC  MEASURELOC                                                       FeatureURL                     geometry
0     35572  22.13280 -159.61995  16010000             Kawaikoi Stream nr Waimea, Kauai, HI       GS  NHD100K          A                    0                  0                    0  http://nwis.waterdata.usgs.gov/nwis/inventory/?site_no=16010000   POINT (-159.61996 22.1328)
1     355

C:\Users\bgorberg\anaconda3\envs\usgs_gis\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'longitude' to 'longitud_1'
  ogr_write(
C:\Users\bgorberg\anaconda3\envs\usgs_gis\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'latitude' to 'latitude_1'
  ogr_write(


In [6]:
gdf

,OBJECTID,Latitude,Longitude,STA_ID,STA_NAME,STA_TYPE,COORDSRC,SITESTATUS,REACH100,MEASURE100,REACH24,MEASURE24,REACHLOC,MEASURELOC,FeatureURL,geometry,longitude,latitude
0,35572,22.13280,-159.61995,16010000,"Kawaikoi Stream nr Waimea, Kauai, HI",GS,NHD100K,A,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-159.61996 22.1328),-159.619964,22.132803
1,35573,22.12525,-159.62140,16011000,"WAIAKOALI STREAM NEAR WAIMEA, KAUAI, HI",GS,NHD100K,I,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-159.62141 22.12525),-159.621414,22.125253
2,35574,22.13357,-159.63015,16012000,"KAUAIKINANA STREAM NEAR WAIMEA, KAUAI, HI",GS,NHD100K,I,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-159.63016 22.13357),-159.630164,22.133573
3,35575,22.11714,-159.60101,16013000,"MOHIHI STREAM AT ALT 3420 FT NR WAIMEA,KAUAI,HI",GS,NHD100K,I,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-159.60102 22.11714),-159.601024,22.117143
4,35576,22.10840,-159.67572,16014000,"KOKEE DITCH NR WAIMEA,KAUAI, HI",GS,NHD100K,I,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-159.67573 22.1084),-159.675734,22.108403
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,36060,21.63361,-158.02000,213801158011201,"NF Elehaha Str at Pupukea Mil Rd, Oahu, HI",LF,SSDB,,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-158.02001 21.63361),-158.020014,21.633613
489,36061,21.67793,-157.95280,214052157572001,"USGS 4 STORM DRAIN AT KAHUKU, OAHU",GS,SSDB,,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-157.95281 21.67793),-157.952814,21.677933
490,36062,21.68098,-157.95590,214103157573101,"USGS 3 KII DITCH AT KAHUKU, OAHU",GS,SSDB,,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-157.95591 21.68098),-157.955914,21.680983
491,36063,21.68848,-157.95250,214130157571901,"USGS 7 DITCH OUTLET AT KAHUKU, OAHU",GS,SSDB,,,0,,0,,0,http://nwis.waterdata.usgs.gov/nwis/inventory/...,POINT (-157.95251 21.68848),-157.952514,21.688483


In [7]:
import requests
import pandas as pd
from io import StringIO


# ============================================================
# CLEAN USGS SITE NUMBERS
# ============================================================

gdf["STA_ID"] = (
    gdf["STA_ID"]
    .astype(str)
    .str.strip()
)


# ============================================================
# GET PEAK-FLOW RECORD FOR ONE SITE
# ============================================================

def get_peak_record(site_no):

    url = "https://waterservices.usgs.gov/nwis/site/"

    params = {
        "format": "rdb",
        "sites": site_no,
        "outputDataTypeCd": "pk"
    }

    try:
        r = requests.get(
            url,
            params=params,
            timeout=30
        )

        r.raise_for_status()

        # Remove comment lines
        lines = [
            line for line in r.text.splitlines()
            if not line.startswith("#")
        ]

        if len(lines) < 3:
            return None, None, None

        df_site = pd.read_csv(
            StringIO("\n".join(lines)),
            sep="\t",
            dtype=str
        )

        # Remove USGS RDB datatype row
        df_site = df_site.iloc[1:].copy()

        if (
            "begin_date" not in df_site.columns
            or "end_date" not in df_site.columns
        ):
            return None, None, None

        start = pd.to_datetime(
            df_site["begin_date"],
            errors="coerce"
        ).min()

        end = pd.to_datetime(
            df_site["end_date"],
            errors="coerce"
        ).max()

        if pd.isna(start) or pd.isna(end):
            return None, None, None

        # Number of calendar years spanned by record
        years = end.year - start.year + 1

        return (
            start.strftime("%Y-%m-%d"),
            end.strftime("%Y-%m-%d"),
            years
        )

    except Exception as e:

        print(f"ERROR {site_no}: {e}")

        return None, None, None


# ============================================================
# QUERY ALL STATIONS
# ============================================================

peak_records = {}

sites = gdf["STA_ID"].dropna().unique()

print(f"Checking {len(sites)} stations...\n")


for i, site in enumerate(sites, start=1):

    start, end, years = get_peak_record(site)

    peak_records[site] = {
        "Peak_Start": start,
        "Peak_End": end,
        "Peak_Years": years
    }

    print(
        f"{i}/{len(sites)} | "
        f"{site} | "
        f"{start} -> {end} | "
        f"{years}"
    )


# ============================================================
# CONVERT RESULTS TO DATAFRAME
# ============================================================

peak_df = (
    pd.DataFrame
    .from_dict(
        peak_records,
        orient="index"
    )
    .reset_index()
    .rename(columns={"index": "STA_ID"})
)


# ============================================================
# JOIN BACK TO GEODATAFRAME
# ============================================================

gdf = gdf.merge(
    peak_df,
    on="STA_ID",
    how="left"
)


# ============================================================
# CHECK RESULTS
# ============================================================

print(
    gdf[
        [
            "STA_ID",
            "STA_NAME",
            "Peak_Start",
            "Peak_End",
            "Peak_Years"
        ]
    ]
)


# How many StreamStats stations actually have peak records?
print(
    "\nStations with peak-flow records:",
    gdf["Peak_Start"].notna().sum()
)

print(
    "Stations without peak-flow records:",
    gdf["Peak_Start"].isna().sum()
)

Checking 493 stations...

1/493 | 16010000 | 1914-09-26 -> 2025-01-26 | 112
ERROR 16011000: 404 Client Error:  for url: https://waterservices.usgs.gov/nwis/site/?format=rdb&sites=16011000&outputDataTypeCd=pk
2/493 | 16011000 | None -> None | None
ERROR 16012000: 404 Client Error:  for url: https://waterservices.usgs.gov/nwis/site/?format=rdb&sites=16012000&outputDataTypeCd=pk
3/493 | 16012000 | None -> None | None
4/493 | 16013000 | 1921-01-16 -> 1971-01-31 | 51
ERROR 16014000: 404 Client Error:  for url: https://waterservices.usgs.gov/nwis/site/?format=rdb&sites=16014000&outputDataTypeCd=pk
5/493 | 16014000 | None -> None | None
ERROR 16015000: 404 Client Error:  for url: https://waterservices.usgs.gov/nwis/site/?format=rdb&sites=16015000&outputDataTypeCd=pk
6/493 | 16015000 | None -> None | None
7/493 | 16016000 | 1916-12-18 -> 1967-12-10 | 52
8/493 | 16017000 | 1919-12-03 -> 1967-11-26 | 49
9/493 | 16018000 | 1916-12-18 -> 1971-01-31 | 56
10/493 | 16019000 | 1921-01-16 -> 2025-04-18

In [10]:
# ============================================================
# SAVE UPDATED FILES
# ==# Remove our duplicate coordinate fields
gdf = gdf.drop(
    columns=["longitude", "latitude"],
    errors="ignore"
)

print(gdf.columns.tolist())
output_shp = output_dir / "Hawaii_StreamStats_Gages_Peak_Record.shp"
output_gpkg = output_dir / "Hawaii_StreamStats_Gages_Peak_Record.gpkg"
output_csv = output_dir / "Hawaii_StreamStats_Gages_Peak_Record.csv"


# Shapefile
gdf.to_file(
    output_shp,
    driver="ESRI Shapefile"
)


# GeoPackage
gdf.to_file(
    output_gpkg,
    layer="streamstats_gages",
    driver="GPKG"
)


# CSV
gdf.drop(
    columns="geometry"
).to_csv(
    output_csv,
    index=False
)


print("\nSaved successfully:")
print(output_shp)
print(output_gpkg)
print(output_csv)

['OBJECTID', 'Latitude', 'Longitude', 'STA_ID', 'STA_NAME', 'STA_TYPE', 'COORDSRC', 'SITESTATUS', 'REACH100', 'MEASURE100', 'REACH24', 'MEASURE24', 'REACHLOC', 'MEASURELOC', 'FeatureURL', 'geometry', 'Peak_Start', 'Peak_End', 'Peak_Years']

Saved successfully:
C:\Users\bgorberg\Documents\Integral\Get_Stream_Gauges\Hawaii_StreamStats_All_Gages\Hawaii_StreamStats_Gages_Peak_Record.shp
C:\Users\bgorberg\Documents\Integral\Get_Stream_Gauges\Hawaii_StreamStats_All_Gages\Hawaii_StreamStats_Gages_Peak_Record.gpkg
C:\Users\bgorberg\Documents\Integral\Get_Stream_Gauges\Hawaii_StreamStats_All_Gages\Hawaii_StreamStats_Gages_Peak_Record.csv


In [11]:
# Extract start and end year from existing dates
gdf["PK_ST_YR"] = pd.to_datetime(
    gdf["Peak_Start"],
    errors="coerce"
).dt.year.astype("Int64")

gdf["PK_END_YR"] = pd.to_datetime(
    gdf["Peak_End"],
    errors="coerce"
).dt.year.astype("Int64")

In [12]:
gdf.to_file(
    output_shp,
    driver="ESRI Shapefile"
)

In [13]:
import requests
import pandas as pd

url = (
    "https://api.waterdata.usgs.gov/"
    "ogcapi/v1/collections/site-types/items"
)

params = {
    "f": "json",
    "limit": 1000
}

r = requests.get(url, params=params, timeout=60)
r.raise_for_status()

data = r.json()

rows = []

for feature in data["features"]:
    p = feature["properties"]

    rows.append({
        "SITE_CODE": feature.get("id", p.get("id")),
        "SITE_TYPE": p.get("site_type_name"),
        "DESCRIPTION": p.get("site_type_description"),
        "PRIMARY": p.get("site_type_primary_flag")
    })

site_types = pd.DataFrame(rows)

# Show anything related to lakes/reservoirs
mask = (
    site_types["SITE_TYPE"].str.contains(
        "lake|reservoir",
        case=False,
        na=False
    )
    |
    site_types["DESCRIPTION"].str.contains(
        "lake|reservoir",
        case=False,
        na=False
    )
)

print(site_types[mask].to_string(index=False))

SITE_CODE                    SITE_TYPE                                                                                                                                                                                                                                                                                                                                                                                   DESCRIPTION PRIMARY
    FA-CI                      Cistern                                                                                                                                        An artificial, non-pressurized reservoir filled by gravity flow and used for water storage. The reservoir may be located above, at, or below ground level. The water may be supplied from diversion of precipitation, surface, or groundwater sources.       N
    FA-CS               Combined sewer                                                                                                        